In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## Part (B) ATE

In [74]:
def generate_ATE():

    # Generate random samples
    N = 1000
    Z = np.random.normal(1, 1, size = (N,))
    R = np.random.normal(0, 0.5, size = (N,))

    sigmoid = lambda x: 1 / (1 + np.exp(-x))

    # 1 denotes T=A, 0 denotes T=B
    T = np.random.binomial(1, sigmoid(Z + R - 0.5), size = (N, ))
    EA = np.random.normal(1.5, 1, size = (N, ))
    EB = np.ones(N) * 0.5
    H = np.random.binomial(1, 1 - sigmoid(Z - EA * T - EB * T), size = (N, ))

    return Z, R, T, H


def construct_ATE(Z, R, T, H):
    df = pd.DataFrame({"Observed Outcome": H, "Treatment": T})

    # Construct 2X2 Table
    T0H0 = df[(df["Treatment"] == 0) & (df["Observed Outcome"] == 0)].count().values[0]
    T0H1 = df[(df["Treatment"] == 0) & (df["Observed Outcome"] == 1)].count().values[0]
    T1H0 = df[(df["Treatment"] == 1) & (df["Observed Outcome"] == 0)].count().values[0]
    T1H1 = df[(df["Treatment"] == 1) & (df["Observed Outcome"] == 1)].count().values[0]

    output_table = pd.DataFrame({"H=0": [T0H0, T1H0], "H=1":[T0H1, T1H1]})
    output_table["T"] = ["T=0", "T=1"]
    output_table.set_index("T", drop=True, inplace= True)

    # Compute ATE
    ATE = output_table.loc["T=1"][1]/output_table.loc["T=1"].sum() \
    - output_table.loc["T=0"][0]/output_table.loc["T=0"].sum()
    return output_table, ATE


def partB():
    Z, R, T, H = generate_ATE()
    return construct_ATE(Z, R, T, H)

In [75]:
ATE_list = []
for i in range(100):
    output_table, ATE = partB()
    ATE_list.append(ATE)
AATE = np.mean(ATE_list)
AATE

0.022023473269386935

In [78]:
output_table, ATE = partB()
output_table, ATE

(     H=0  H=1
 T            
 T=0  233  168
 T=1  215  384,
 0.06002106586621925)

## Part(C) Confounders Adjustment

In [86]:
def adjusting_sickness():
    # Z > 0 is sick, Z < 0 is healthy
    # R > 0 is openness, R < 0 is stubborn
    # T = 1 treatment, T = 0 control
    # H = 1 positive ourcome, H = 0 negative outcome
    Z, R, T, H = generate_ATE()


    # Convert to boolean array, for easy computation
    ZB = (Z > 0).astype(int)
    RB = (R > 0).astype(int)


    # Adjusting for confounders Z 

    # P (ZB = 1)
    psick = (Z > 0).astype(int).sum() / Z.shape[0]

    # Calculate ATE
    df = pd.DataFrame({"Observed Result":H, "Treatment":T, "Sick":ZB})

    # Treatment
    treatment = psick * df[(df["Sick"] == 1) & (df["Treatment"] == 1)]["Observed Result"].mean() +\
                (1 - psick) * df[(df["Sick"] == 0) & (df["Treatment"] == 1)]["Observed Result"].mean()

    # Control
    control = psick * df[(df["Sick"] == 1) & (df["Treatment"] == 0)]["Observed Result"].mean() +\
                (1 - psick) * df[(df["Sick"] == 0) & (df["Treatment"] == 0)]["Observed Result"].mean()

    return treatment - control

In [94]:
adjusting_sickness()

0.31706584916322234